In [18]:
#importar librerias
import json
import copy
import matplotlib.pyplot as plt
import numpy as np

In [90]:
#función para generar gráficas
def graficar_(experimentos, resultados, guardar, ver=False):
    for i in range(len(experimentos)):
        try:
            with open(experimentos[i], "r", encoding="utf-8") as a:
                base = json.load(a)
        except:
            print("NO SE ENCONTRÓ ", experimentos[i])
            continue

        try:
            with open(resultados[i], "r", encoding="utf-8") as f:
                res = json.load(f)
        except:
            print("NO SE ENCONTRÓ ", resultados[i])
            continue

        fig, axs = plt.subplots(2, 2, figsize=(10, 8))
        

        #---------------------------------------------------------
        axs[0, 0].plot([i for i in range(len(res["Best loss"]))], res["Best loss"], label=f"Mejor loss ({min(res["Best loss"])})")
        axs[0, 0].plot([i for i in range(len(res["Worst loss"]))], res["Worst loss"], label=f"Peor loss ({max(res["Worst loss"])})")
        axs[0, 0].set_xlabel("Iteraciones")
        axs[0, 0].set_ylabel("Loss ("+base["Fitness"]+")")
        axs[0, 0].set_title("Loss obtenido en "+resultados[i])
        axs[0, 0].grid(True)
        axs[0, 0].legend(loc="upper right")

        #---------------------------------------------------------
        axs[0, 1].plot([i for i in range(len(res["Best accuracy"]))], res["Best accuracy"], label=f"Mejor accuracy ({max(res["Best accuracy"])})")
        axs[0, 1].plot([i for i in range(len(res["Worst accuracy"]))], res["Worst accuracy"], label=f"Peor accuracy ({min(res["Worst accuracy"])})")
        axs[0, 1].set_xlabel("Iteraciones")
        axs[0, 1].set_ylabel("Accuracy")
        axs[0, 1].set_title("Accuracy obtenido en "+resultados[i])
        axs[0, 1].grid(True)
        axs[0, 1].legend(loc="upper right")

        #---------------------------------------------------------
        superior = np.array(res["Promedio"]) + np.array(res["Desvest"])
        inferior = np.array(res["Promedio"]) - np.array(res["Desvest"])
        axs[1, 0].plot([i for i in range(len(res["Best iteration"]))], res["Best iteration"], label=f"Mejor fitness ({min(res["Best iteration"])})")
        axs[1, 0].plot([i for i in range(len(res["Worst iteration"]))], res["Worst iteration"], label=f"Peor fitness ({max(res["Worst iteration"])})")
        axs[1, 0].plot([i for i in range(len(res["Promedio"]))], res["Promedio"], linewidth=1.5, color="teal", label="Valor promedio\npor iteracion")                # línea fuerte
        axs[1, 0].plot([i for i in range(len(superior))], superior, linestyle='--', linewidth=1.5, alpha=0.5, color="teal", label="Percentiles\n25 y 75")     # borde inferior
        axs[1, 0].plot([i for i in range(len(inferior))], inferior, linestyle='--', linewidth=1.5, alpha=0.5, color="teal")     # borde superior
        axs[1, 0].fill_between([i for i in range(len(superior))], superior, inferior, alpha=0.2, color="teal", label="±1σ")
        axs[1, 0].set_xlabel("Iteraciones")
        axs[1, 0].set_ylabel("Fitness")
        axs[1, 0].set_title("Fitness obtenido en "+resultados[i])
        axs[1, 0].grid(True)
        axs[1, 0].legend(loc="upper right")

        #---------------------------------------------------------
        axs[1, 1].plot([i for i in range(len(res["Succes"]))], res["Succes"], label="Succes")
        axs[1, 1].set_xlabel("Iteraciones")
        axs[1, 1].set_ylabel("Succes")
        axs[1, 1].set_title("Succes (si o no, 1 o 0)")
        axs[1, 1].grid()
        axs[1, 1].legend(loc="upper right")

        #---------------------------------------------------------
        prom=np.mean(np.array(res["Tiempo por iteración"]))
        tot=sum(np.array(res["Tiempo por iteración"]))
        fig.suptitle(f"Resultados para "+resultados[i]+f"\n{prom:.2f} segundos por iteracion\n{tot:.2f} segundos en total\npruebas en datos de: \""+base["Reporte"]+"\"")
        plt.tight_layout()
        plt.savefig(guardar[i], dpi=300)
        if(ver==True):
            plt.show()
        else:
            plt.close()

def graficar(ruta, ver=False):
    with open(ruta, "r", encoding="utf-8") as a:
        base = json.load(a)

    experimentos=[]
    resultados=[]
    graficas=[]

    for i in range(base["N archivos"]):
        with open(base["Archivos"][i], "r", encoding="utf-8") as f:
                archvio=json.load(f)

        experimentos.append(base["Archivos"][i])
        resultados.append(archvio["Archivo salida"])
        graficas.append(archvio["Archivo graficas"])

    graficar_(experimentos, 
              resultados, 
              graficas,
              ver)

if __name__=="__main__":
    graficar("../j_7.json")

In [96]:
#crear los jsons de los experimentos
seed_base=277010
dataset="micro_mnist"
optimizador="SHADE"

Fs=[[0.8 for i in range(1)]]

Crs=[[0.9  for i in range(1)]]

extras=[{"Data type":        "double",
         "Dataset":          dataset, 
         "N iters":          1000, 
         "N inds":           200, 
         "N experimentos":   1, 
         "Seed":             seed_base+i%10, 
         "Archivo salida":   "", #--------------------------------------
         "Archivo graficas": "", #--------------------------------------
         "Optimizador":      optimizador,
         "Lower bound":      -1.0, 
         "Upper bound":      1.0, 
         "Fitness":          "accuracy", 
         "F":                0.5, #--------------------------------------
         "C_r":              0.5, #--------------------------------------
         "N capas":          4,
         "Estructura":       [784, 128, 64, 10],
         "Activaciones":     ["ReLU", "ReLU", "ReLU", "nada"],
         "Verbose":          -1, 
         "Reporte":          "test",
         "Clip":             1, 
         "Normalizacion":    1,
         "Mutacion":         "pbest"}
         for i in range(len(Fs)*len(Fs[0]))]
     
count=0
for i in range(len(Fs)):
    for j in range(len(Fs[i])):
        extras[count]["Archivo salida"]="../resultados/"+dataset+f"_{i}_{j}_"+extras[count]["Fitness"]+"_"+optimizador+".txt"
        extras[count]["Archivo graficas"]="../graficas/"+dataset+f"_{i}_{j}_"+extras[count]["Fitness"]+"_"+optimizador+".png"

        extras[count]["F"]=Fs[i][j]
        extras[count]["C_r"]=Crs[i][j]

        print(count, 
              "../resultados/"+dataset+f"_{i}_{j}_"+extras[count]["Fitness"]+"_"+optimizador+".txt", 
              "../graficas/"+dataset+f"_{i}_{j}_"+extras[count]["Fitness"]+"_"+optimizador+".png", 
              Fs[i][j], 
              Crs[i][j])

        count+=1

count=0
nombre_archivo=[]
for i in range(len(Fs)):
    for j in range(len(Fs[i])):
        nombre_archivo.append(f"../experimentos/j_{i}_{j}_"+extras[count]["Fitness"]+"_"+dataset+"_"+optimizador+".json")
        count+=1

for i, nuevo_json in enumerate(extras):
    with open(nombre_archivo[i], "w", encoding="utf-8") as f:
        json.dump(nuevo_json, f, indent=4, ensure_ascii=False)


iguales=[[], [], [], [], [], []]
count=0
for i in range(len(Fs)):
    for j in range(len(Fs[i])):
        iguales[i].append(f"../experimentos/j_{i}_{j}_"+extras[count]["Fitness"]+"_"+dataset+"_"+optimizador+".json")
        count+=1

data = {
    "N archivos": len(nombre_archivo),
    "N hilos":    7,
    "Archivos":   nombre_archivo,
    "Iguales":    iguales
}

# Writing to sample.json
with open("../j_6_SHADE.json", "w") as h:
    json.dump(data, h, indent=4)

0 ../resultados/micro_mnist_0_0_accuracy_SHADE.txt ../graficas/micro_mnist_0_0_accuracy_SHADE.png 0.8 0.9


In [1]:
import fitz
import os

carpeta_pdf = "../papers"
carpeta_txt = "/Users/imauriciolopez/Downloads/papers_txt"

os.makedirs(carpeta_txt, exist_ok=True)

for archivo in os.listdir(carpeta_pdf):

    if archivo.endswith(".pdf"):

        ruta_pdf = os.path.join(carpeta_pdf, archivo)

        ruta_txt = os.path.join(carpeta_txt, archivo.replace(".pdf", ".txt"))

        doc = fitz.open(ruta_pdf)

        texto = ""

        for page in doc:

            texto += page.get_text() + "\n"

        with open(ruta_txt, "w", encoding="utf-8") as f:

            f.write(texto)

In [5]:
import math as mt
n=10
mat=[[j*n+i for i in range(n)] for j in range(n)]
mat_2=[i for i in range(n*n)]
mat_3=[[0 for i in range(n)] for j in range(n)]

for i in range(n*n):
    mat_3[i%n][mt.floor(i/n)]=mat_2[i]
mat_3

[[0, 10, 20, 30, 40, 50, 60, 70, 80, 90],
 [1, 11, 21, 31, 41, 51, 61, 71, 81, 91],
 [2, 12, 22, 32, 42, 52, 62, 72, 82, 92],
 [3, 13, 23, 33, 43, 53, 63, 73, 83, 93],
 [4, 14, 24, 34, 44, 54, 64, 74, 84, 94],
 [5, 15, 25, 35, 45, 55, 65, 75, 85, 95],
 [6, 16, 26, 36, 46, 56, 66, 76, 86, 96],
 [7, 17, 27, 37, 47, 57, 67, 77, 87, 97],
 [8, 18, 28, 38, 48, 58, 68, 78, 88, 98],
 [9, 19, 29, 39, 49, 59, 69, 79, 89, 99]]

In [34]:
n=2
for i in range(n*n):
    for j in range(n*n):
        print(mt.floor(j/n)+mt.floor(i/n)*n, ",", j%n+(i%n)*n, end="|")
    print()

0 , 0|0 , 1|1 , 0|1 , 1|
0 , 2|0 , 3|1 , 2|1 , 3|
2 , 0|2 , 1|3 , 0|3 , 1|
2 , 2|2 , 3|3 , 2|3 , 3|
